# 01 — Loss demonstration walk-through

Short walk-through of the three loss-side primitives that this skeleton
fixes: Bühlmann-Straub credibility weighting (exposure form), the three
concrete likelihood classes, and the Credence-State loss assembly. The
companion toy demonstration on linear Gaussian state-space (see the
appendix material with the SPRIND submission) provides the full plot-
and-explain version. This notebook is a smoke pass to confirm the
package wires up end-to-end after `uv pip install -e .`.

Cells below are intentionally minimal; the goal is *one screenful per
primitive*, not a tutorial.

In [1]:
import torch
from bayesian_native_ssm.losses.buhlmann import buhlmann_z, buhlmann_weighted_loss
from bayesian_native_ssm.losses.likelihoods import (
    gauss_likelihood, huber_likelihood, wasserstein_likelihood,
)
from bayesian_native_ssm.losses.trust_posterior import trust_posterior_loss

## Bühlmann-Z, exposure form

Rare sub-populations get small `Z_c`, which pulls the local loss toward
the global mean. Common sub-populations get `Z_c ≈ 1`, retaining local
autonomy.

In [2]:
w = torch.tensor([0.01, 1.0, 10.0, 100.0])
z = buhlmann_z(w, k=1.0)
for wi, zi in zip(w.tolist(), z.tolist()):
    print(f'w={wi:>6.2f}   Z={zi:.4f}')

w=  0.01   Z=0.0099
w=  1.00   Z=0.5000
w= 10.00   Z=0.9091
w=100.00   Z=0.9901


## Three likelihood classes side by side

On a 100σ outlier, the Gauss NLL explodes quadratically; the Huber loss
stays linear past the threshold δ. Wasserstein-2 compares full
distributions on a shared 1-D support normalised to [0, 1].

In [3]:
base = torch.zeros(10)
outlier = torch.zeros(10); outlier[0] = 100.0
print('gauss :', gauss_likelihood(outlier, base, sigma=1.0).item())
print('huber :', huber_likelihood(outlier, base, delta=1.0).item())

p = torch.softmax(torch.randn(16), dim=0)
q = torch.softmax(torch.randn(16), dim=0)
print('wasserstein (p vs q):', wasserstein_likelihood(p, q).item())
print('wasserstein (p vs p):', wasserstein_likelihood(p, p.clone()).item())

gauss : 5009.189453125
huber : 99.5
wasserstein (p vs q): 0.025439053773880005
wasserstein (p vs p): 0.015775032341480255


## Credence-State loss assembly

Combine token cross-entropy (`L_data`), the dimension-factorised
reliability loss (`L_trust`, mapping to `L_credence` + `L_update` in
the five-component v5 decomposition), and an optional Bühlmann-Straub
credibility weighting across sub-population contexts. The function
name `trust_posterior_loss` is the legacy code symbol; see the
repository README for the naming map.

In [4]:
B, T, V = 4, 8, 32
model_output = torch.randn(B, T, V)
data_targets = torch.randint(0, V, (B, T))
ts = {'calibration': torch.randn(B), 'honesty': torch.randn(B)}
tt = {'calibration': torch.zeros(B), 'honesty': torch.zeros(B)}
contexts = torch.tensor([0, 0, 1, 1])  # two sub-populations

out = trust_posterior_loss(
    model_output, data_targets, ts, tt,
    context_assignments=contexts,
    likelihood_class='gauss',
    lambdas={'calibration': 1.0, 'honesty': 0.5},
    buhlmann_k=1.0,
)
for k, v in out.items():
    print(f'{k:>8s}: {v.item():.4f}')

  L_data: 133.6313
 L_trust: 6.8857
 L_prior: 0.0000
 L_total: 140.5170
